# 06 · Selection Bias, PriDe und LRZ

Dieses Notebook verbindet die Methodik aus 04–05 mit der ausführbaren Analyse.
Alle Zahlenbeispiele sind **synthetisch**, keine Modellresultate. Es lädt weder
Modelle noch Datensätze aus dem Internet und reicht keine GPU-Jobs ein.

**Ablauf:** vorverarbeitete Fragen → First-Token-Scoring → Baseline und vier
Permutationen → Prior auf D_e → PriDe → Vergleichstabelle.
Wir evaluieren vorhandene Modelle; es findet kein Training oder Fine-Tuning statt.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for base in candidates for p in (base, base / "analysis")
                     if (p / "src").is_dir() and (p / "pyproject.toml").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Notebook im MCQ-Projekt öffnen.")
for path in (PROJECT_ROOT, PROJECT_ROOT / "scripts"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print("Projekt:", PROJECT_ROOT)


Projekt: /Users/diergexiaohai/Desktop/MCQ/analysis


## 1 · First Token: Wie entsteht eine Antwort?

Nach dem fertigen Prompt liefert das Modell Logits für das nächste Token.
Der echte Scorer liest die vier Token-IDs für **A, B, C, D** aus und normalisiert
nur über diese vier Kandidaten:

$$p_i = \frac{\exp(z_i)}{\sum_{j\in\{A,B,C,D\}}\exp(z_j)}.$$

Das größte p_i bestimmt die Antwort. Dies ist eine auf A–D beschränkte Verteilung,
nicht die Wahrscheinlichkeit über das gesamte Vokabular. Es wird kein längerer
Antworttext generiert. Die Token-Diagnostik prüft die Buchstaben im tatsächlichen
Prompt-Kontext; ein isoliertes `encode("A")` genügt dafür nicht.

Implementation: `src/models/token_scoring.py` (`FirstTokenOptionScorer`).


In [2]:
labels = list("ABCD")
logits = np.array([2.0, 1.4, 0.2, -0.1])  # Nur ein Rechenbeispiel!
probabilities = np.exp(logits - logits.max())
probabilities /= probabilities.sum()
display(pd.DataFrame({"Option": labels, "Logit (Demo)": logits,
                      "p (normalisiert über A–D)": probabilities}))
print("Vorhersage:", labels[int(probabilities.argmax())])


,Option,Logit (Demo),p (normalisiert über A–D)
0,A,2.0,0.544494
1,B,1.4,0.298825
2,C,0.2,0.090004
3,D,-0.1,0.066677


Vorhersage: A


## 2 · Selection Bias: Bevorzugt das Modell eine Position?

Die Baseline verwendet die ursprüngliche Reihenfolge. Danach werden vier
zyklische Reihenfolgen geprüft, einschließlich der ursprünglichen. Jede
Antwort steht dadurch einmal an jeder Position. Das Goldlabel muss mitwandern.

Die Metriken umfassen Accuracy, Antwortverteilung, Recall für jede Goldposition
und **RStd = Standardabweichung der vier positionsbezogenen Recalls**.
Der Permutation-Spread ist die größte minus die kleinste Accuracy der vier
Reihenfolgen. Eine ungleiche Antwortverteilung allein beweist noch keinen Bias.
Bei fehlenden Goldklassen ist RStd nicht bestimmbar (`NaN`).


In [3]:
contents = ["Inhalt 1", "Inhalt 2", "Inhalt 3", "Inhalt 4"]
gold_content = 1
rows = []
for shift in range(4):
    order = [(position + shift) % 4 for position in range(4)]
    rows.append({"Permutation": shift, **{label: contents[index]
                 for label, index in zip(labels, order)},
                 "Richtiges Label": labels[order.index(gold_content)]})
display(pd.DataFrame(rows))


,Permutation,A,B,C,D,Richtiges Label
0,0,Inhalt 1,Inhalt 2,Inhalt 3,Inhalt 4,B
1,1,Inhalt 2,Inhalt 3,Inhalt 4,Inhalt 1,A
2,2,Inhalt 3,Inhalt 4,Inhalt 1,Inhalt 2,D
3,3,Inhalt 4,Inhalt 1,Inhalt 2,Inhalt 3,C


## 3 · PriDe: Positionsprior schätzen und korrigieren

Die Experimentkonfiguration legt `alpha=0.05` und Seed 42 fest. Aus der ausgewählten
Population werden nach Sortieren der IDs reproduzierbar Fragen für D_e gezogen.
Die Anzahl wird abgerundet, mit mindestens einer Frage. D_r enthält die übrigen
Fragen. D_e ist eine Kalibrierungsmenge, kein Trainingssplit für Modellgewichte.

Pro Frage in D_e wird aus den vier zyklischen Vorhersagen ein Positionsprior
gebildet (normalisiertes geometrisches Mittel). Die individuellen Priors werden
zum globalen Prior gemittelt. Goldlabels werden für diese Schätzung nicht benutzt.
Für D_r gilt anschließend:

$$p_i^{\mathrm{PriDe}} = \frac{p_i / \pi_i}{\sum_j p_j / \pi_j}.$$

Die implementierte Policy verwendet auf **D_e die zyklisch zusammengeführte
Inhaltsvorhersage**, auf **D_r die globale Prior-Korrektur**. Die Gesamtmetrik
fasst beide Gruppen zusammen; sie ist daher kein reiner D_r-Vergleich.


In [4]:
from src.pride.prior_estimation import (
    estimation_sample_count, estimate_sample_prior, estimate_global_prior,
)
from src.pride.debias import apply_pride_correction
from src.utils.config import named_config

experiment = named_config("experiments", "thesis")
alpha = experiment["pride"]["alpha"]
display(pd.DataFrame([
    {"Population (Demo)": n, "D_e": estimation_sample_count(n, alpha),
     "D_r": n - estimation_sample_count(n, alpha)} for n in [8, 100, 1000]
]))
# Beispiel mit einer konstanten Positionspräferenz über vier Permutationen.
permutation_probabilities = np.tile([0.50, 0.20, 0.20, 0.10], (4, 1))
sample_prior = estimate_sample_prior(permutation_probabilities)
prior = estimate_global_prior([sample_prior])
observed = np.array([0.40, 0.30, 0.20, 0.10])
corrected = apply_pride_correction(observed, prior)
display(pd.DataFrame({"Option": labels, "Prior (Demo)": prior,
                      "Vorher": observed, "Nachher": corrected}))
print("Vorher:", labels[observed.argmax()], "→ Nachher:", labels[corrected.argmax()])


,Population (Demo),D_e,D_r
0,8,1,7
1,100,5,95
2,1000,50,950


,Option,Prior (Demo),Vorher,Nachher
0,A,0.5,0.4,0.186047
1,B,0.2,0.3,0.348837
2,C,0.2,0.2,0.232558
3,D,0.1,0.1,0.232558


Vorher: A → Nachher: B


## 4 · Welche Dateien führen die Analyse aus?

| Datei | Aufgabe |
|---|---|
| `scripts/run_baseline.py` | Ursprüngliche Antwortreihenfolge auswerten |
| `scripts/run_permutations.py` | Vier zyklische Reihenfolgen und deren Metriken |
| `scripts/estimate_prior.py` | D_e auswählen und Prior speichern |
| `scripts/run_pride.py` | Gespeicherten Prior anwenden; Vorher/Nachher vergleichen |
| `scripts/run_pipeline.py` | Alle vier Schritte verbinden und wiederaufnehmen |
| `scripts/run_campaign.py` | Versuchsreihe pro Modell × Datensatz planen und einreichen |
| `scripts/aggregate_results.py` | Zugehörige Summaries zu Vergleichstabellen verbinden |

Die Kernmodelle werden aus der aktuellen Konfiguration gelesen; Qwen3 war bislang
zusätzliches Testmodell. Ein erfolgreicher Zwei-Fragen-Test ist ein technischer
Nachweis, keine belastbare Messung von Accuracy oder Bias.


In [5]:
display(pd.DataFrame([{"Kernmodell": alias} for alias in experiment["core_models"]]))
print("Prior-Anteil:", alpha)
print("Sampling-Seed:", experiment["pride"]["primary_seed"])
print("D_e-Policy:", experiment["pride"]["estimation_sample_policy"])


,Kernmodell
0,llama-3.1-8b-instruct
1,qwen2.5-7b-instruct
2,gemma-2-9b-it


Prior-Anteil: 0.05
Sampling-Seed: 42
D_e-Policy: cyclic


## 5 · Was ist eine Slurm-Datei?

Eine `.sbatch`-Datei ist ein Arbeitsauftrag für LRZ. `#SBATCH`-Zeilen bestellen
Ressourcen, zum Beispiel eine GPU, Arbeitsspeicher und maximale Laufzeit.
Danach folgen die Befehle zum Aktivieren der Python-Umgebung und zum Starten des
Analyseskripts. `sbatch` reicht den Auftrag ein; Slurm entscheidet, wann er startet.

- **Login Node:** Dateien übertragen, Umgebung einrichten, Jobs einreichen.
- **Compute Node mit GPU:** Slurm führt dort den Modelllauf aus.
- **Umgebung `mcq-analysis`:** Python mit den benötigten Paketen.
- **Partition:** Gruppe von Rechnern, z. B. mit A100-GPUs; aktuelle Namen per `sinfo` prüfen.
- **`LIMIT=8`:** acht Fragen; bei den Permutationen daraus 32 Prompts.
- **`PD (Priority)`:** Auftrag wartet; daraus lässt sich keine verlässliche Wartezeit ablesen.

Die nächste Zelle **zeigt nur den Dateiinhalt** und führt ihn nicht aus.


In [6]:
print((PROJECT_ROOT / "scripts/slurm/run_analysis_step.sbatch").read_text())


#!/bin/bash
#SBATCH --job-name=mcq-analysis
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=8
#SBATCH --mem=96G
#SBATCH --time=08:00:00
#SBATCH --output=outputs/logs/slurm/%x-%j.out
#SBATCH --error=outputs/logs/slurm/%x-%j.err
set -euo pipefail
: "${SLURM_JOB_ID:?Submit with sbatch; do not run on the login node}"
cd "$SLURM_SUBMIT_DIR"
source scripts/slurm/environment.sh
hostname
scontrol show job "$SLURM_JOB_ID"
nvidia-smi
python -c 'import torch; assert torch.cuda.is_available(), "CUDA unavailable"'
MODEL="${MODEL:-qwen2.5-7b-instruct}"
DATASET="${DATASET:-medqa}"
STEP="${STEP:-pipeline}"
args=(--model "$MODEL" --dataset "$DATASET" --batch-size "${BATCH_SIZE:-4}" --scorer real)
[[ -z "${LIMIT:-}" ]] || args+=(--limit "$LIMIT")
[[ -z "${RESUME:-}" ]] || args+=(--resume "$RESUME")
ARTIFACT="${PREPROCESSED:-$(python -c 'import sys; sys.path.insert(0,"scripts"); from run_pipeline import artifact_for; print(artifact_for(sys.argv[1]))' "$DATASET")}"
args+=(--preprocessed "$ARTIFACT")
case "$S

## 6 · Ausführen und anschließend auswerten

Die vollständige Anleitung steht in [scripts/slurm/README.md](../scripts/slurm/README.md).
Auf LRZ werden zunächst kleine echte Pipelines geprüft, anschließend volle Läufe.
Notebook 07 liest ausschließlich die gespeicherten Ergebnisse.

`outputs/raw/` enthält Einzelvorhersagen und Metadaten; `outputs/summaries/` die
Metriken. In `outputs/pipelines/<id>/` stehen der Ablaufstatus, die Schritt-Logs,
der gespeicherte Prior und `comparison.csv`. Abgebrochene Läufe lassen sich über
`--resume` mit unveränderten Einstellungen fortsetzen.

Keine Preprocessing-Wiederholung ist dafür nötig. Für Thesis-Ergebnisse sind
vollständige Läufe und die Prüfung von Jobstatus, Token-Diagnostik, Stichprobengröße
und Metadaten erforderlich. Mock-Ergebnisse bleiben separat gekennzeichnet.


## 7 · Originalvergleich und RQ1/RQ2

Die Referenzdaten von Zheng et al. sind separat unter `data/reference/zheng2024`
gespeichert: MMLU (13.592 nach Originalfilter), ARC-Challenge (1.165) und
CommonsenseQA (1.216). CSQA behält alle **fünf** Antwortoptionen; unsere aktuelle
Vier-Optionen-Pipeline muss dafür noch erweitert werden. Die vorhandenen
medizinischen Artefakte wurden nicht verändert.

Ein numerischer Vergleich mit dem fixierten Originalcode bestätigt die
Vier-Optionen-Rechnung für die geprüften positiven Wahrscheinlichkeiten.
Prompt, Sampling/Wiederholungen und Filter unterscheiden sich noch; dies ist
keine Behauptung einer vollständigen experimentellen Replikation.

Als Arbeitsstruktur bietet sich an: **RQ1** Replikation mit neueren Modellen auf
den Originalbenchmarks; **RQ2** Erweiterung auf MedQA und MedMCQA. Die endgültige
Nummerierung und Formulierung sind mit dem Thesis-Entwurf abzugleichen.

Details, Quellen und offene Implementationspunkte:
[Originalvergleich und Datenaudit](../docs/zheng2024_replikation.md).
